In [1]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=".env")  # Charger les variables depuis .env

api_key = os.getenv("API_KEY")


In [2]:
url='https://api.openweathermap.org/data/2.5/forecast?'


In [3]:
import requests
import json
import pandas as pd
import datetime
import plotly.express as px
import plotly.graph_objects as go

In [4]:
def get_weather(lat,lon):
  payload={'appid':api_key,'lat':lat,'lon':lon,'units':'metric'}
  response=requests.get(url,params=payload)
  return response.json()


In [5]:
def score_temperature(T):
    if 18 <= T <= 25:
        return 1
    else:
        return max(0,1 - abs(T - 18) / 10) if T<18 else max(0,1 - abs(T -25) / 10)

def score_humidite(H):
    if 30 <= H <= 60:
        return 1
    else:
        return max(0,1 - abs(H - 30) / 60) if H<30 else max(0,1 - abs(H - 60) / 60)

def score_couverture_nuageuse(CN):
    if 20 <= CN <= 50:
        return 1
    else:
        return max(0,1 - abs(CN - 20) / 50) if CN<20 else max(0,1 - abs(CN - 50) / 50)

def score_vent(V):
    if 5 <= V <= 15:
        return 1
    else:
        return max(0,1 - abs(V - 5) / 20) if V<5 else max(0,1 - abs(V - 15) / 20)
def score_precipitations(R):
    if R == 0:
        return 1  # Pas de précipitations
    elif R <= 2:
        return max(0.8, 1 - R / 5)  # Légère pluie, score légèrement réduit
    else:
        return max(0, 1 - R / 10)  # Pluie modérée à forte, score réduit

def coefficient_confort_meteorologique(T, H, CN, V,R):
    score_T = score_temperature(T)
    score_H = score_humidite(H)
    score_CN = score_couverture_nuageuse(CN)
    score_V = score_vent(V)
    score_Precip = score_precipitations(R)


    CCM = (0.35 * score_T + 0.2 * score_H + 0.15 * score_CN +
           0.15 * score_V + 0.15 * score_Precip)

    return CCM



In [6]:
def calcul_CCM(lat,lon):
  data=get_weather(lat,lon)
  data_weather=[{'day':datetime.datetime.fromtimestamp(x['dt']),
                 'hour':datetime.datetime.fromtimestamp(x['dt']).hour,
                 'clouds': x['clouds']['all'],
                 'temperature':x['main']['temp'],
                 'humidity':x['main']['humidity'],
                 'wind':x['wind']['speed'],
                 'rain_mm': x.get('rain', {}).get('3h', 0)
                 } for x in data['list']]
  df_weather=pd.DataFrame(data_weather)
  df_weather['CCM']=df_weather.apply(lambda x: coefficient_confort_meteorologique(x['temperature'],x['humidity'],x['clouds'],x['wind'],x['rain_mm']),axis=1)
  return df_weather[(df_weather['hour']>=9) & (df_weather['hour']<=18)]['CCM'].median()


In [7]:
len(get_weather(43.296174,5.369953)['list'])*3/24


5.0

In [8]:
df_cities_latlon=pd.read_csv('cities_lat_long.csv').rename(columns={'Unnamed: 0':'city'})

In [9]:
df_cities_latlon['CCM']=(df_cities_latlon.apply(lambda x: calcul_CCM(x['lat'],x['lon']),axis=1)).round(3)

In [10]:
# df_cities_latlon.reset_index(drop=False,inplace=True,names='index')
df_cities_latlon.sort_values(by='CCM',ascending=False,inplace=True)

df_cities_latlon.to_csv('cities_lat_long_ccm.csv')